# Fake Reviews Dataset Setup

This notebook downloads the Kaggle dataset `mexwell/fake-reviews-dataset` into a local project folder, then loads it for quick inspection and summary statistics.

## Kaggle API Setup (One-Time)

If you see `Kaggle credentials not found at ~/.kaggle/kaggle.json`, do this once:

1. Sign in to Kaggle: https://www.kaggle.com
2. Open Account settings: https://www.kaggle.com/settings/account
3. In **API**, click **Create Legacy API Key** (downloads `kaggle.json`).
4. In a terminal, run:

```bash
mkdir -p ~/.kaggle
mv ~/Downloads/kaggle.json ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json
```

5. Verify file exists (either location works in this notebook):

```bash
ls -l ~/.kaggle/kaggle.json
ls -l ./.kaggle/kaggle.json
```

6. Re-run this notebook.


In [1]:
%pip install -q --upgrade kaggle pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import pandas as pd

DATASET_SLUG = "mexwell/fake-reviews-dataset"





DATA_DIR = Path("data/raw/" + DATASET_SLUG.split("/")[-1])
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle authentication is required.
home_kaggle = Path.home() / ".kaggle" / "kaggle.json"
project_kaggle = Path.cwd() / ".kaggle" / "kaggle.json"

if home_kaggle.exists():
    os.environ["KAGGLE_CONFIG_DIR"] = str(home_kaggle.parent)
elif project_kaggle.exists():
    os.environ["KAGGLE_CONFIG_DIR"] = str(project_kaggle.parent)
    print(f"Using Kaggle credentials from: {project_kaggle}")
else:
    print("Kaggle credentials not found.")
    print("Expected one of:")
    print(f"- {home_kaggle}")
    print(f"- {project_kaggle}")
    raise FileNotFoundError("Create kaggle.json and rerun this cell.")

kaggle_executable = shutil.which("kaggle")
if kaggle_executable:
    download_cmd = [
        kaggle_executable,
        "datasets",
        "download",
        "-d",
        DATASET_SLUG,
        "-p",
        str(DATA_DIR),
        "--unzip",
    ]
else:
    # Fallback for environments where only the python package is exposed.
    download_cmd = [
        sys.executable,
        "-m",
        "kaggle.cli",
        "datasets",
        "download",
        "-d",
        DATASET_SLUG,
        "-p",
        str(DATA_DIR),
        "--unzip",
    ]

csv_files = sorted(DATA_DIR.rglob("*.csv"))
if not csv_files:
    print(f"Downloading {DATASET_SLUG} to {DATA_DIR} ...")
    result = subprocess.run(download_cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Kaggle download failed:\n{result.stderr or result.stdout}")
    csv_files = sorted(DATA_DIR.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR}.")

dataset_path = csv_files[0]
print(f"Using dataset file: {dataset_path}")

Using dataset file: data/raw/fake-reviews-dataset/fake reviews dataset.csv


In [3]:
df = pd.read_csv(dataset_path)

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head(10))

Rows: 40,432 | Columns: 4


,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...
5,Home_and_Kitchen_5,3.0,CG,I WANTED DIFFERENT FLAVORS BUT THEY ARE NOT.
6,Home_and_Kitchen_5,5.0,CG,They are the perfect touch for me and the only...
7,Home_and_Kitchen_5,3.0,CG,These done fit well and look great. I love th...
8,Home_and_Kitchen_5,5.0,CG,"Great big numbers & easy to read, the only thi..."
9,Home_and_Kitchen_5,5.0,CG,My son loves this comforter and it is very wel...


In [4]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null_count": df.notna().sum(),
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(dropna=True),
})

summary["sample_values"] = df.apply(
    lambda s: ", ".join(s.dropna().astype(str).unique()[:3])
)

numeric_stats = df.describe(include=["number"]).T
summary = summary.join(numeric_stats, how="left")

display(summary)

,dtype,non_null_count,null_count,null_pct,unique_count,sample_values,count,mean,std,min,25%,50%,75%,max
category,str,40432,0,0.0,10,"Home_and_Kitchen_5, Sports_and_Outdoors_5, Ele...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rating,float64,40432,0,0.0,5,"5.0, 1.0, 3.0",40432.0,4.256579,1.144354,1.0,4.0,5.0,5.0,5.0
label,str,40432,0,0.0,2,"CG, OR",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
text_,str,40432,0,0.0,40412,"Love this! Well made, sturdy, and very comfor...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
